# Extract oMST REC and LDI scores using subject ID

This notebook scans a JATOS results zip or extracted folder, finds each `data.txt`,
uses the `subject` field inside the TXT as the participant ID,
extracts the final `REC` and `LDI` values from the oMST summary,
and saves a combined CSV.


In [4]:
from pathlib import Path
import zipfile
import re
import pandas as pd

INPUT_PATH = Path('/Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set/oMST/jatos_results_data_20261018031655.zip')
OUTPUT_DIR = Path('/Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set/oMST/data')
OUTPUT_CSV = OUTPUT_DIR / 'combined_oMST_result_by_subject.csv'

SUBJECT_PATTERN = re.compile(r'"subject"\s*:\s*"([^"]+)"')
REC_LDI_PATTERN = re.compile(r'REC,\s*([0-9]*\.?[0-9]+)\s*,\s*LDI,\s*([0-9]*\.?[0-9]+)')


In [5]:
def list_data_files(input_path: Path):
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(input_path, 'r') as zf:
            return [name for name in zf.namelist() if name.endswith('data.txt')]
    elif input_path.is_dir():
        return sorted(input_path.rglob('data.txt'))
    else:
        raise FileNotFoundError(f'Cannot find input: {input_path}')

def read_text(input_path: Path, file_ref):
    if input_path.is_file() and input_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(input_path, 'r') as zf:
            return zf.read(file_ref).decode('utf-8', errors='replace')
    return Path(file_ref).read_text(encoding='utf-8', errors='replace')

def extract_subject(text: str):
    matches = SUBJECT_PATTERN.findall(text)
    return matches[-1] if matches else None

def extract_rec_ldi(text: str):
    matches = REC_LDI_PATTERN.findall(text)
    if not matches:
        return None
    rec, ldi = matches[-1]
    return float(rec), float(ldi)


In [6]:
rows = []
skipped = []

for file_ref in list_data_files(INPUT_PATH):
    text = read_text(INPUT_PATH, file_ref)
    subject = extract_subject(text)
    score = extract_rec_ldi(text)

    if subject is None or score is None:
        skipped.append(str(file_ref))
        continue

    rec, ldi = score
    rows.append({
        'participant_id': subject,
        'REC': rec,
        'LDI': ldi,
    })

df = pd.DataFrame(rows).drop_duplicates(subset=['participant_id']).sort_values('participant_id').reset_index(drop=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Saved {len(df)} rows to: {OUTPUT_CSV}')
print(f'Skipped files: {len(skipped)}')
df.head(10)


Saved 48 rows to: /Users/jianleguo/Documents/GitHub/vmps_mario/data/batch02_new_set/oMST/data/combined_oMST_result_by_subject.csv
Skipped files: 192


,participant_id,REC,LDI
0,5bbc0004ac45e6000131d380,0.650,0.832
1,5c476621b521ee00019abf67,0.800,0.666
2,5c58e01e6f4ec5000164c35a,1.000,0.719
3,5e6b034bffc1a90c3fe8c234,0.900,0.810
4,5e84bd7c34d5d9072128fac1,0.950,0.712
5,5ee92b925f0f482bb1e73668,0.969,0.621
6,5ef6308c40846d2dfca0b66a,0.634,0.497
7,615ddab1e4f013092538b6c5,0.700,0.700
8,62877d5f6a054e53313ac916,1.000,0.754
9,62c84ca82b03c9aedbca4a2a,0.800,0.641


In [7]:
df


,participant_id,REC,LDI
0,5bbc0004ac45e6000131d380,0.650,0.832
1,5c476621b521ee00019abf67,0.800,0.666
2,5c58e01e6f4ec5000164c35a,1.000,0.719
3,5e6b034bffc1a90c3fe8c234,0.900,0.810
4,5e84bd7c34d5d9072128fac1,0.950,0.712
5,5ee92b925f0f482bb1e73668,0.969,0.621
6,5ef6308c40846d2dfca0b66a,0.634,0.497
7,615ddab1e4f013092538b6c5,0.700,0.700
8,62877d5f6a054e53313ac916,1.000,0.754
9,62c84ca82b03c9aedbca4a2a,0.800,0.641
